In [2]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from random import randint
from random import random
import math
import csv
from collections import deque
from collections import Counter

In [3]:
#Generate Grid with random edge lengths
def GenGrid(l,w,r):
    Grid = [[0 for k in range(l*w)] for y in range(l*w)]
    for i in range(l*w):
        for j in range(w*l):
            if Grid[j][i] != 0:
                Grid[i][j] = Grid[j][i]
            else:
                if ((j == i - 1) or (j == i + 1) or (j == i + w) or (j == i - w)):
                    Grid[i][j] = round((r[1]-r[0]) * random() + r[0],2)
                if ((i+1)%w == 0 and j == i + 1) or (i%w == 0 and j == i - 1):
                    Grid[i][j] = 0
    return Grid

In [4]:
def GenMap(f):
    Edges = [[],[],[]]
    Lines = []
    for i in f:
        Lines.append(i[0:-1])
    for i in range(2):
        for j in range(len(Lines)):
            if Lines[j][1] == ' ':
                Edges[i].append(int(Lines[j][0]))
                Lines[j] = Lines[j][2:]
            else:
                Edges[i].append(int(Lines[j][0:2]))
                Lines[j] = Lines[j][3:]
    for j in Lines:
        if len(j) == 1:
            Edges[2].append(int(j[0]))
        else:
            Edges[2].append(int(j[0:2]))
    Grid = [[0 for i in range(len(Edges[0])+1)] for j in range(len(Edges[0])+1)]
    for i in range(len(Edges[0])):
        Val1 = Edges[0][i]
        Val2 = Edges[1][i]
        Grid[Val1][Val2] = Edges[2][i]
        Grid[Val2][Val1] = Edges[2][i]
    return Grid

In [5]:
def VisualiseGraphs(adj_matrix, paths,l,w,highlighted):
    G = nx.Graph()
    n = len(adj_matrix)

    # Build base graph
    for i in range(n):
        for j in range(i + 1, n):
            if adj_matrix[i][j] != 0:
                G.add_edge(i, j, weight=adj_matrix[i][j])

    pos = {}
    for i in range(n):
        x = i % w
        y = -(i // w) 
        pos[i] = (x, y)

    nx.draw(
        G,
        pos,
        node_color="lightgray",
        with_labels=True,
        node_size=300,
        font_size=12,
        edge_color="gray"
    )

    edge_labels = nx.get_edge_attributes(G, 'weight')
    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

    edge_labels = nx.get_edge_attributes(G, 'weight')
    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

    legend_elements = []

    shared_nodes = set()
    if paths:
        counts = Counter(node for route in paths for node in route)
        shared_nodes = {node for node, count in counts.items() if count > 1}

    if paths:
        edge_colors = ["red", "blue", "green", "purple", "brown"]
        node_colors = ["yellow", "cyan", "orange", "pink", "lime"]

        for idx, route in enumerate(paths):
            if len(route) > 1:

                route_edges = list(zip(route, route[1:]))

                DG = nx.DiGraph()
                DG.add_edges_from(route_edges)

                edge_color = edge_colors[idx % len(edge_colors)]
                node_color = node_colors[idx % len(node_colors)]

                rad = 0.1 * ((idx % 3) - 1)

                nx.draw_networkx_edges(
                    DG,
                    pos,
                    edge_color=edge_color,
                    width=3,
                    alpha=0.8,
                    arrows=True,
                    arrowstyle='-|>',
                    arrowsize=20,
                )

                normal_nodes = [n for n in route if n not in shared_nodes]

                nx.draw_networkx_nodes(
                    G,
                    pos,
                    nodelist=normal_nodes,
                    node_color=node_color,
                    node_size=400,
                    edgecolors="black"
                )

                legend_elements.append(
                    Line2D([0], [0], color=edge_color, lw=3, label=f"Route {idx+1}"))

    if shared_nodes:
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=list(shared_nodes),
            node_color="limegreen",
            node_size=500,
            edgecolors="black",
            linewidths=2
        )

        legend_elements.append(
            Line2D([0], [0], marker='o', color='w',
                   label='Transfer Stop',
                   markerfacecolor='limegreen',
                   markeredgecolor='black',
                   markersize=10)
        )
    nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=highlighted,
            node_color="crimson",
            node_size=500,
            edgecolors="black",
            linewidths=2
        )
    legend_elements.append(Line2D([0], [0], marker='o', color='w', label='Route 1 Stop',
               markerfacecolor='yellow', markersize=10,markeredgecolor='black'))
    legend_elements.append(Line2D([0], [0], marker='o', color='w', label='Route 2 Stop',
               markerfacecolor='cyan', markersize=10,markeredgecolor='black'))
    if legend_elements:
        plt.legend(handles=legend_elements, loc='right')
    plt.axis('equal')
    plt.savefig("Graph3.png")
    plt.show()

In [6]:
def NearestNode(distances, CheckedNode):
    best = [0,9999]
    for i in distances:
        if i not in CheckedNode and distances[i] < best[1]:
            best = [i,distances[i]]
    return best

In [7]:
def CheckNeighbours(adj_matrix,path,Stops,Node,CheckedNode,distances,UniqStops):
    
    CheckedNode.append(Node[0])   
    if Node[0] in UniqStops:
        Stops.append(Node[0])
    
    for i in range(len(adj_matrix)):
        dist = adj_matrix[Node[0]][i]
        if dist != 0 and not i in CheckedNode:
            if i in distances:
                if Node[1] + dist < distances[i]:
                    distances[i] = Node[1] + dist
            else:
                distances[i] = Node[1] + dist
    return distances, Stops, CheckedNode

In [8]:
def NearestStop(adj_matrix,paths,Node,k2,k3,path,UniqStops,TransferStops,Useless):
    if Node in path:
        BusDis = 0
        for i in range(len(path)-path.index(Node)-1):
            BusDis += k3/k4*(adj_matrix[path[-i-1]][path[-i-2]])
        return Node, 0, round(BusDis,3), 0  #Start node, Walk time, Totalcost, Transfer
    CheckedNode = []
    distances = {Node : 0}
    Stops = []
    
    while len(Stops) < len(UniqStops):
        Nearest = NearestNode(distances,CheckedNode)
        distances, Stops, CheckedNode = CheckNeighbours(adj_matrix, path, Stops, Nearest, CheckedNode, distances,UniqStops)

    #Bus distance calc
    PathA = paths.index(path)
    PathB = (PathA + 1) % 2
    BusDisA = [0]
    for i in range(len(path)-1):
        BusDisA.insert(0,adj_matrix[path[-(i+1)]][path[-i-2]] + BusDisA[0])
    
    pathcost = [[],[]]   #stop, cost
    for i in path:
        pathcosti = k2/k6 * distances[i] + k3/k4 * BusDisA[path.index(i)]
        pathcost[0].append(i)
        pathcost[1].append(pathcosti)
    
    BusDisB = [0]
    Useful = []
    count = 0
    if len(TransferStops[PathB]) > 0:
        while not TransferStops[PathB][-1] in Useful:
            Useful.append(paths[PathB][count])
            count += 1
        for i in range(len(Useful)-1):
            BusDisB.insert(0,adj_matrix[Useful[-(i+1)]][Useful[-i-2]] + BusDisB[0])

        for i in Useful:
            if not i in Useless[PathB]:
                pathcosti = k2/k6 * distances[i] + k3/k4 * BusDisB[Useful.index(i)] + k3/k4 * BusDisA[path.index(TransferStops[PathB][-1])]
                pathcost[0].append(i)
                pathcost[1].append(pathcosti)
    
    
    cost = min(pathcost[1])
    EndStop = pathcost[0][pathcost[1].index(cost)]
    
    if not EndStop in path:
        return EndStop, distances[EndStop]/k6, round(cost,3), 1
    return EndStop, distances[EndStop]/k6, round(cost,3), 0

In [9]:
def BusTimetable(adj_matrix,paths,k4,k5,Headways,n):
    Timetables = []
    for k in range(len(paths)):
        Buses = int(n/Headways[k]) + 1
        Timetable = [{0 + i*Headways[k] :paths[k][0]} for i in range(Buses)]
        time = 0
        for i in range(1,len(paths[k])):
            time += k5 + int(adj_matrix[paths[k][i]][paths[k][i-1]] / k4) + 1
            for j in range(Buses):
                Timetable[j][time  + j * Headways[k]] = paths[k][i]
        Timetables.append(Timetable)
    return Timetables

In [10]:
def GenSeed(adj_matrix,n,SpawnDensity):
    Seed = [[],[],[],[]]         #Stop, Time, Quantity, Destination
    for i in range(len(paths)):
        for j in range(len(SpawnDensity[i])):
            Cumul = 0
            for k in range(1,n):
                Cumul += SpawnDensity[i][j]/60
                if Cumul >= 1:
                    Seed[0].append(j)
                    Seed[1].append(k)
                    Seed[2].append(int(Cumul))
                    Seed[3].append(i)
                    Cumul -= int(Cumul)
    return Seed

In [22]:
def Loading(adj_matrix,paths,k2,k3,Headways,n,SpawnDensity):
    
    UniqStops = []
    for i in (paths[0] + paths[1]):
        if not i in UniqStops:
            UniqStops.append(i)
    
    TransferStops = [[],[]] #Transfer stops in order for path[i]
    for i in paths[0]:
        if i in paths[1]:
            TransferStops[0].append(i)
    for i in paths[1]:
        if i in paths[0]:
            TransferStops[1].append(i)
    
    Useless = [[],[]] #If you are transfering from stop i, it is useless to get on Useless[i] stops
    for i in range(2):
        TransfersLeft = len(TransferStops[0])
        for j in paths[i]:
            if j in TransferStops[0] or TransfersLeft <= 0:
                Useless[i].append(j)
                TransfersLeft -= 1
    StopData = [[i for i in range(len(adj_matrix))], [NearestStop(adj_matrix,paths,j,k2,k3,paths[0],UniqStops,TransferStops,Useless) for j in range(len(adj_matrix))],
                [NearestStop(adj_matrix,paths,j,k2,k3,paths[1],UniqStops,TransferStops,Useless) for j in range(len(adj_matrix))]]
    #Start stop, Nearest stop (Des 1), Nearest stop (des 2)
    #Nearest Stop: (EndStop, Cost (incl bus), Transfer (0/1))
    Timetable = BusTimetable(adj_matrix,paths,k4,k5,Headways,n)
    Seed = GenSeed(adj_matrix,n,SpawnDensity)

    return StopData,Timetable, Seed

In [11]:
def RunRoute(adj_matrix,paths,StopData,Timetables,Seed,headway,BusCap,Time):
    PassengerStops = [ [paths[0], [[0,0] for i in range(len(paths[0]))], [deque() for i in range(len(paths[0]))] ] ,
                       [paths[1], [[0,0] for i in range(len(paths[1]))], [deque() for i in range(len(paths[1]))]] ]
    PassengerTimes = [[],[],[],[]]     #Stop, Time, Quantity, Destination

    Locations = [[i for i in range(len(adj_matrix))],[0 for j in range(len(adj_matrix))],
                [0 for j in range(len(adj_matrix))]]
                #Stop, Pass (des 1), Pass (des 2) 
    BusLoad = [[[0,0] for i in range(len(Timetables[0]))],[ [0,0] for j in range(len(Timetables[1]))]]
    WalkandBusTime = 0
    WaitTimes = []
    for t in range(Time):
        #Spawn in new passengers
        if t in Seed[1]:
            #print(t)
            for i in range(len(Seed[0])):
                if t == Seed[1][i]:
                    if Seed[3][i] == 0:
                        Locations[1][Seed[0][i]] += Seed[2][i]
                    else:
                        Locations[2][Seed[0][i]] += Seed[2][i]
        
        #Move passengers from their start points onto passengertimes
        
        if max(Locations[1]) > 0:
            for i in range(len(Locations[0])):
                if Locations[1][i] > 0:
                    NearStop0 = StopData[1][i]
                    PassengerTimes[0].append(NearStop0[0])
                    if NearStop0[1] == 0:
                        PassengerTimes[1].append(t)
                    else:
                        PassengerTimes[1].append(math.ceil((t + NearStop0[1])))
                        WalkandBusTime += NearStop0[2]
                    PassengerTimes[2].append(Locations[1][i])
                    PassengerTimes[3].append(0)
                    Locations[1][i] = 0
                    
                
                if Locations[2][i] > 0:
                    NearStop1 = StopData[2][i]
                    PassengerTimes[0].append(NearStop1[0])
                    if NearStop1[1] == 0:
                        PassengerTimes[1].append(t)
                    else:
                        PassengerTimes[1].append(math.ceil((t + NearStop1[1])))
                        WalkandBusTime += NearStop1[2]
                    PassengerTimes[2].append(Locations[2][i])
                    Locations[2][i] = 0
                    PassengerTimes[3].append(1)

        #Passengers reach the bus stops
        if t in PassengerTimes[1]:
            for i in range(len(PassengerTimes[1])):
                if PassengerTimes[1][i] == t and PassengerTimes[0][i] in paths[0]:
                    Stop = PassengerTimes[0][i]
                    StopIndex = paths[0].index(Stop)
                    if PassengerTimes[3][i] == 0:
                        PassengerStops[0][1][StopIndex][0] += PassengerTimes[2][i]
                        PassengerStops[0][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0
                    elif not PassengerTimes[0][i] in paths[1]:
                        PassengerStops[0][1][StopIndex][1] += PassengerTimes[2][i]
                        PassengerStops[0][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0
                        
                elif PassengerTimes[1][i] == t and PassengerTimes[0][i] in paths[1]:
                    Stop = PassengerTimes[0][i]
                    StopIndex = paths[1].index(PassengerTimes[0][i])
                    if PassengerTimes[3][i] == 1:
                        PassengerStops[1][1][StopIndex][1] += PassengerTimes[2][i]
                        PassengerStops[1][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0
                    elif not PassengerTimes[0][i] in paths[0]:
                        PassengerStops[1][1][StopIndex][0] += PassengerTimes[2][i]
                        PassengerStops[1][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0
                    
        #Bus picks up the passengers
        for j in range(2):
            for i in Timetables[j]:
                if t in i.keys():
                    Stop = i[t]
                    StopNumber = PassengerStops[j][0].index(Stop)
                    #Bus unloads passengers:
                    if Stop in paths[0] and Stop in paths[1]:
                        OtherRoute = (j+1) % 2
                        OtherStopNumber = PassengerStops[OtherRoute][0].index(Stop)
                        for k in range(BusLoad[j][Timetables[j].index(i)][OtherRoute]):
                            PassengerStops[OtherRoute][2][OtherStopNumber].append(t)
                        PassengerStops[OtherRoute][1][OtherStopNumber][OtherRoute] += \
                        BusLoad[j][Timetables[j].index(i)][OtherRoute]

                        BusLoad[j][Timetables[j].index(i)][OtherRoute] = 0
                        BusLoad[j][Timetables[j].index(i)][(j+1)%2] = 0
        
                    #Bus loads passengers:
                    if sum(BusLoad[j][Timetables[j].index(i)]) + sum(PassengerStops[j][1][StopNumber]) > BusCap:
                        queue = PassengerStops[j][2][StopNumber]
                        Space = BusCap - sum(BusLoad[j][Timetables[j].index(i)])
                        BusLoad[j][Timetables[j].index(i)][j] += Space
                        PassengerStops[j][1][StopNumber][j] -= Space
                        for k in range(min(Space,len(queue))):
                            WaitTimes.append(t - queue.popleft())
                            
                    else:
                        BusLoad[j][Timetables[j].index(i)][j] += PassengerStops[j][1][StopNumber][j]
                        BusLoad[j][Timetables[j].index(i)][(j+1)%2] += PassengerStops[j][1][StopNumber][(j+1)%2]
                        queue = PassengerStops[j][2][StopNumber]

                        local = PassengerStops[j][1][StopNumber][j]
                        transfer = PassengerStops[j][1][StopNumber][(j+1)%2]

                        total_waiting = local + transfer
                        current_load = sum(BusLoad[j][Timetables[j].index(i)])
                        space = BusCap - current_load

                        boarding = min(space, total_waiting)

                        for _ in range(boarding):
                            WaitTimes.append(t - queue.popleft())
                        PassengerStops[j][1][StopNumber][j] = 0
                        PassengerStops[j][1][StopNumber][(j+1)%2] = 0
                    
    #Bus cost calc
    BusTime = 0
    for i in range(2):
        for j in Timetables[i]:
            if max(j) < Time:
                BusTime += max(Timetables[i][0])
            else:
                BusTime += Time - min(j)

    WaitTime = sum(WaitTimes)

    print('Individual Costs \n',
          'Walk and Bus Time Cost', round(WalkandBusTime,3), '\n',
          'Wait Time Cost', round(k7 * WaitTime,3),'\n',
          'Bus Cost', round(k1 * BusTime,3), '\n',
          'Total Cost', round((WalkandBusTime) + k7 * WaitTime + k1 * BusTime ,3))
    Pass0 = 0
    for i in BusLoad[0]:
        Pass0 += i[0]
    print('Passengers served (Bus 0) = ', Pass0)
    Pass1 = 0
    for i in BusLoad[1]:
        Pass1 += i[1]
    print('Passengers served (Bus 1) = ', Pass1)
    print(BusLoad)

In [12]:
def SilentRunRoute(adj_matrix,paths,StopData,Timetables,Seed,Headways,BusCap,Time):
    PassengerStops = [ [paths[0], [[0,0] for i in range(len(paths[0]))], [deque() for i in range(len(paths[0]))] ] ,
                       [paths[1], [[0,0] for i in range(len(paths[1]))], [deque() for i in range(len(paths[1]))]] ]
    PassengerTimes = [[],[],[],[]]     #Stop, Time, Quantity, Destination

    Locations = [[i for i in range(len(adj_matrix))],[0 for j in range(len(adj_matrix))],
                [0 for j in range(len(adj_matrix))]]
                #Stop, Pass (des 1), Pass (des 2) 
    BusLoad = [[[0,0] for i in range(len(Timetables[0]))],[ [0,0] for j in range(len(Timetables[1]))]]
    WalkandBusTime = 0
    WaitTimes = []
    for t in range(Time):
        if t in Seed[1]:
            for i in range(len(Seed[0])):
                if t == Seed[1][i]:
                    if Seed[3][i] == 0:
                        Locations[1][Seed[0][i]] += Seed[2][i]
                    else:
                        Locations[2][Seed[0][i]] += Seed[2][i]
        
        #Move passengers from their start points onto passengertimes
        if max(Locations[1]) > 0:
            for i in range(len(Locations[0])):
                if Locations[1][i] > 0:
                    NearStop0 = StopData[1][i]
                    PassengerTimes[0].append(NearStop0[0])
                    if NearStop0[1] == 0:
                        PassengerTimes[1].append(t)
                    else:
                        PassengerTimes[1].append(math.ceil((t + NearStop0[1])))
                        WalkandBusTime += NearStop0[2]
                    PassengerTimes[2].append(Locations[1][i])
                    PassengerTimes[3].append(0)
                    Locations[1][i] = 0
                    
                if Locations[2][i] > 0:
                    NearStop1 = StopData[2][i]
                    PassengerTimes[0].append(NearStop1[0])
                    if NearStop1[1] == 0:
                        PassengerTimes[1].append(t)
                    else:
                        PassengerTimes[1].append(math.ceil((t + NearStop1[1])))
                        WalkandBusTime += NearStop1[2]
                    PassengerTimes[2].append(Locations[2][i])
                    Locations[2][i] = 0
                    PassengerTimes[3].append(1)

        #Passengers reach the bus stops
        if t in PassengerTimes[1]:
            for i in range(len(PassengerTimes[1])):
                if PassengerTimes[1][i] == t and PassengerTimes[0][i] in paths[0]:
                    Stop = PassengerTimes[0][i]
                    StopIndex = paths[0].index(Stop)
                    if PassengerTimes[3][i] == 0:
                        PassengerStops[0][1][StopIndex][0] += PassengerTimes[2][i]
                        PassengerStops[0][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0
                    elif not PassengerTimes[0][i] in paths[1]:
                        PassengerStops[0][1][StopIndex][1] += PassengerTimes[2][i]
                        PassengerStops[0][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0
                        
                elif PassengerTimes[1][i] == t and PassengerTimes[0][i] in paths[1]:
                    Stop = PassengerTimes[0][i]
                    StopIndex = paths[1].index(PassengerTimes[0][i])
                    if PassengerTimes[3][i] == 1:
                        PassengerStops[1][1][StopIndex][1] += PassengerTimes[2][i]
                        PassengerStops[1][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0
                    elif not PassengerTimes[0][i] in paths[0]:
                        PassengerStops[1][1][StopIndex][0] += PassengerTimes[2][i]
                        PassengerStops[1][2][StopIndex].append(t)
                        PassengerTimes[2][i] = 0

        #Bus picks up the passengers
        for j in range(2):
            for i in Timetables[j]:
                if t in i.keys():
                    Stop = i[t]
                    StopNumber = PassengerStops[j][0].index(Stop)
                    #Bus unloads passengers:
                    if Stop in paths[0] and Stop in paths[1]:
                        OtherRoute = (j+1) % 2
                        OtherStopNumber = PassengerStops[OtherRoute][0].index(Stop)
                        for k in range(BusLoad[j][Timetables[j].index(i)][OtherRoute]):
                            PassengerStops[OtherRoute][2][OtherStopNumber].append(t)
                        PassengerStops[OtherRoute][1][OtherStopNumber][OtherRoute] += \
                        BusLoad[j][Timetables[j].index(i)][OtherRoute]

                        BusLoad[j][Timetables[j].index(i)][OtherRoute] = 0
                        BusLoad[j][Timetables[j].index(i)][(j+1)%2] = 0
        
                    #Bus loads passengers:
                    if sum(BusLoad[j][Timetables[j].index(i)]) + sum(PassengerStops[j][1][StopNumber]) > BusCap:
                        queue = PassengerStops[j][2][StopNumber]
                        Space = BusCap - sum(BusLoad[j][Timetables[j].index(i)])
                        BusLoad[j][Timetables[j].index(i)][j] += Space
                        PassengerStops[j][1][StopNumber][j] -= Space
                        for k in range(min(Space,len(queue))):
                            WaitTimes.append(t - queue.popleft())
                    else:
                        BusLoad[j][Timetables[j].index(i)][j] += PassengerStops[j][1][StopNumber][j]
                        BusLoad[j][Timetables[j].index(i)][(j+1)%2] += PassengerStops[j][1][StopNumber][(j+1)%2]
                        queue = PassengerStops[j][2][StopNumber]

                        local = PassengerStops[j][1][StopNumber][j]
                        transfer = PassengerStops[j][1][StopNumber][(j+1)%2]

                        total_waiting = local + transfer
                        current_load = sum(BusLoad[j][Timetables[j].index(i)])
                        space = BusCap - current_load

                        boarding = min(space, total_waiting)

                        for _ in range(boarding):
                            WaitTimes.append(t - queue.popleft())
                        PassengerStops[j][1][StopNumber][j] = 0
                        PassengerStops[j][1][StopNumber][(j+1)%2] = 0
    #Bus cost calc
    BusTime = 0
    for i in range(2):
        for j in Timetables[i]:
            if max(j) < Time:
                BusTime += max(Timetables[i][0])
            else:
                BusTime += Time - min(j)

    WaitTime = sum(WaitTimes)
    TotalCost = round((WalkandBusTime) + k7 * WaitTime + k1 * BusTime ,3)
    return TotalCost



In [13]:
def OptHeadway(adj_matrix,paths,Time,Range):
    TotalCosts = []
    BusCosts = []
    PassengerCosts = []
    RangeDiff = Range[1]-Range[0]
    for i in range(Range[0],Range[1]):
        for j in range(Range[0],Range[1]):
            Headways = [i,j]
            StopData,Timetables,Seed = Loading(adj_matrix,paths,k2,k3,Headways,n,SpawnDensity)
            Total = SilentRunRoute(adj_matrix,paths,StopData,Timetables,Seed,Headways,BusCap,n)
            TotalCosts.append(Total)
    Min = min(TotalCosts)
    OptIndex = TotalCosts.index(Min)
    OptHeadway = (int(OptIndex/RangeDiff) + Range[0], (OptIndex % RangeDiff) + Range[0], Min)
    return OptHeadway

In [14]:
def FindNeighbours2(adj_matrix, Node, Checked):
    Neighbours = []
    for i in range(len(adj_matrix)):
        if adj_matrix[Node][i] != 0 and not i in Checked:
            Neighbours.append(i)
    return Neighbours

In [15]:
def MutualNeighbours(adj_matrix, Node1, Node2):
    Neigh1 = FindNeighbours2(adj_matrix, Node1, [])
    Neigh2 = FindNeighbours2(adj_matrix, Node2, [])
    Mutual = []
    for i in Neigh1:
        for j in Neigh2:
            if i == j:
                Mutual.append(i)
    return Mutual

In [17]:
def BackwardsSearch3(adj_matrix, AltPath, NewDes):
    paths = [AltPath, [NewDes]]
    Checked = [NewDes]
    
    StopData,Timetables,Seed = Loading(adj_matrix,paths,k2,k3,Headways,n,SpawnDensity)
    CurrentCost = SilentRunRoute(adj_matrix,paths,StopData,Timetables,Seed,Headways,BusCap,n)
    NewCost = CurrentCost - 1
    
    total = 0
    AllCosts = []
    AllOptions = []
    while NewCost <= CurrentCost or total < 10:
        CurrentCost = NewCost
        Neigh = FindNeighbours2(adj_matrix,paths[1][0],Checked)
        Costs = []
        Options = []
        if Neigh == []:
            break
        for i in range(len(Neigh)):
            NewRoute = paths[1][:]
            NewRoute.insert(0,Neigh[i])
            total += 1
            StopData,Timetables,Seed = Loading(adj_matrix,[AltPath, NewRoute],k2,k3,Headways,n,SpawnDensity)
            NewCost = SilentRunRoute(adj_matrix,[AltPath, NewRoute],StopData,Timetables,Seed,Headways,BusCap,n)
            Costs.append(NewCost)
            Options.append(NewRoute)
            AllCosts.append(NewCost)
            AllOptions.append(NewRoute)
            Checked.append(Neigh[i])

        if CurrentCost < min(Costs) and total > 10:
            break
        else:
            paths[1] = Options[Costs.index(min(Costs))]
            NewCost = min(Costs)
            Checked.append(paths[1][0])
    FinalRoute = AllOptions[AllCosts.index(min(AllCosts))]

    print('Total routes checked', total)
    return (FinalRoute, min(AllCosts))

In [18]:
l = 8
w = 8
r = [1,1]
adj_matrix = GenGrid(l,w,r)

In [19]:
k1 = 50/60   #Bus operating cost
k2 = 10/60    #Walk cost (passenger access value)
k3 = 5/60   #Bus travel cost (value of in-vehicle time)
k4 = 20/60    # Bus Speed in km/minute 
k5 = 1     # Stop time in minutes
k6 = 3/60 # Passenger walk speed in km/minute
k7 = 10/60 #Wait time cost
Headways = [9,9]    #Bus headway in minutes
BusCap = 60   #Bus Capacity
n = 300        #Time the bus runs for

In [20]:
SpawnDensity = [[4] *l*w, [4] *(l*w)]
destinations = [l*w-1, l*w - l]

In [23]:
paths = [[23, 15, 14, 13, 12, 11, 10, 9, 17, 25, 33, 41, 42, 43, 44, 45, 46, 47, 55, 63],
         [54, 53, 52, 44, 43, 42, 41, 40, 48, 56]]
StopData,Timetables,Seed = Loading(adj_matrix,paths,k2,k3,Headways,n,SpawnDensity)

In [24]:
RunRoute(adj_matrix,paths,StopData,Timetables,Seed,Headways,BusCap,n)

Individual Costs 
 Walk and Bus Time Cost 10777.503 
 Wait Time Cost 2331.5 
 Bus Cost 3435.833 
 Total Cost 16544.836
Passengers served (Bus 0) =  1154
Passengers served (Bus 1) =  822
[[[60, 0], [60, 0], [60, 0], [46, 0], [42, 0], [22, 0], [44, 0], [43, 0], [41, 0], [42, 0], [22, 0], [44, 0], [43, 0], [41, 0], [42, 0], [22, 0], [44, 0], [43, 0], [41, 0], [42, 0], [22, 0], [44, 0], [43, 0], [40, 0], [38, 0], [18, 0], [30, 0], [24, 0], [18, 21], [14, 16], [6, 8], [8, 10], [5, 7], [0, 2]], [[0, 8], [0, 5], [0, 7], [0, 24], [0, 21], [0, 21], [0, 29], [0, 31], [0, 25], [0, 30], [0, 30], [0, 32], [0, 32], [0, 26], [0, 30], [0, 30], [0, 32], [0, 32], [0, 26], [0, 30], [0, 30], [0, 32], [0, 32], [0, 26], [0, 30], [0, 30], [0, 32], [0, 32], [0, 26], [0, 27], [0, 5], [0, 6], [4, 7], [0, 6]]]


In [230]:
Range = [5,10]
OptHeadway(adj_matrix,paths,n,Range)

(9, 9, 16544.836)

In [25]:
AltPath = [23, 15, 14, 13, 12, 11, 10, 9, 17, 25, 33, 41, 42, 43, 44, 45, 46, 47, 55, 63]
NewDes = 56
Route, Cost = BackwardsSearch3(adj_matrix, AltPath, NewDes)
print(Route)
print(Cost)

Total routes checked 13
[50, 42, 41, 40, 48, 56]
18255.522


In [26]:
paths = [[23, 15, 14, 13, 12, 11, 10, 9, 17, 25, 33, 41, 42, 43, 44, 45, 46, 47, 55, 63],
         [54, 53, 52, 44, 43, 42, 41, 40, 48, 56]]

In [ ]:
VisualiseGraphs(adj_matrix, paths,l,w,[])